In [2]:
# Block preprocessing

import pandas as pd
import os
import glob

save_path = (
    "/lakehouse/default/Files/data/"
    "processed/deals/block_processed.csv"
)

os.makedirs(
    "/lakehouse/default/Files/data/processed/deals",
    exist_ok=True
)

def clean_block(files, source):

    data = []

    for file in files:

        try:
            df = pd.read_csv(file)

        except pd.errors.EmptyDataError:

            print(
                "Skipped empty file:",
                os.path.basename(file)
            )
            continue

        if df.empty:
            continue

        df.columns = (
            df.columns
            .str.strip()
            .str.replace('"', '', regex=False)
            .str.replace('\ufeff', '', regex=False)
        )

        if source == "NSE":

            df = df.rename(columns={

                "Date": "date",

                "Symbol": "symbol",

                "SecurityName": "security_name",
                "ClientName": "client_name",
                "Buy/Sell": "deal_type",
                "QuantityTraded": "quantity",
                "TradePrice/Wght.Avg.Price": "price",

                "Security Name": "security_name",
                "Client Name": "client_name",
                "Buy / Sell": "deal_type",
                "Quantity Traded": "quantity",
                "Trade Price / Wght. Avg. Price": "price"
            })

        else:

            df = df.rename(columns={
                "Deal Date": "date",
                "Deal_Date": "date",
                "Security Code": "symbol",
                "Security_Code": "symbol",
                "Company": "security_name",
                "Client Name": "client_name",
                "Client_Name": "client_name",
                "Deal Type": "deal_type",
                "Deal_Type": "deal_type",
                "Quantity": "quantity",
                "Price": "price"
            })

        keep_cols = [
            "date",
            "symbol",
            "security_name",
            "client_name",
            "deal_type",
            "quantity",
            "price"
        ]

        df = df[keep_cols]

        df["quantity"] = pd.to_numeric(
            df["quantity"]
            .astype(str)
            .str.replace(",", "", regex=False),
            errors="coerce"
        )

        df["price"] = pd.to_numeric(
            df["price"],
            errors="coerce"
        )

        df["deal_type"] = (
            df["deal_type"]
            .astype(str)
            .str.upper()
            .replace({
                "B": "BUY",
                "P": "BUY",
                "BUY": "BUY",
                "S": "SELL",
                "SELL": "SELL"
            })
        )

        df["date"] = pd.to_datetime(
            df["date"],
            errors="coerce",
            dayfirst=True
        )

        df["source"] = source

        data.append(df)

    if len(data) == 0:
        return pd.DataFrame()

    return pd.concat(
        data,
        ignore_index=True
    )

final_df = pd.concat([

    clean_block(
        glob.glob(
            "/lakehouse/default/Files/data/raw/nse_block_hist/*.csv"
        ),
        "NSE"
    ),

    clean_block(
        glob.glob(
            "/lakehouse/default/Files/data/raw/nse_block_inc/*.csv"
        ),
        "NSE"
    ),

    clean_block(
        glob.glob(
            "/lakehouse/default/Files/data/raw/bse_block_hist/*.csv"
        ),
        "BSE"
    ),

    clean_block(
        glob.glob(
            "/lakehouse/default/Files/data/raw/bse_block_inc/*.csv"
        ),
        "BSE"
    )

], ignore_index=True)

final_df = final_df.dropna(
    subset=[
        "date",
        "security_name",
        "deal_type"
    ]
)

final_df = final_df.drop_duplicates()

final_df = final_df.sort_values(
    ["date", "security_name"]
)

final_df.to_csv(
    save_path,
    index=False
)

print("SUCCESS")
print("Rows:", len(final_df))
print("Saved:", save_path)

StatementMeta(, 4b8c811d-62f7-4cb6-ae24-68fcc3931f36, 4, Finished, Available, Finished, False)

Skipped empty file: block_02-06-2026.csv
Skipped empty file: block_05-06-2026.csv
Skipped empty file: block_06-06-2026.csv
Skipped empty file: block_07-06-2026.csv
Skipped empty file: block_09-06-2026.csv
SUCCESS
Rows: 14479
Saved: /lakehouse/default/Files/data/processed/deals/block_processed.csv


In [3]:
block_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "Files/data/processed/deals/block_processed.csv"
    )
)

spark.sql(
    "DROP TABLE IF EXISTS block_daily"
)

(
    block_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("block_daily")
)

print("SUCCESS")
print("Rows:", block_df.count())

block_df.printSchema()

display(
    block_df.limit(10)
)

StatementMeta(, 4b8c811d-62f7-4cb6-ae24-68fcc3931f36, 5, Finished, Available, Finished, False)

SUCCESS
Rows: 14479
root
 |-- date: date (nullable = true)
 |-- symbol: string (nullable = true)
 |-- security_name: string (nullable = true)
 |-- client_name: string (nullable = true)
 |-- deal_type: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- source: string (nullable = true)



SynapseWidget(Synapse.DataFrame, a38c0de0-9709-4ed7-a8f9-90c0bba9f976)